In [ ]:
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
# Load data
df = pd.read_csv('IMDB Dataset.csv')
x = df['review'].values
y = df['sentiment'].map({'positive': 1, 'negative': 0}).values

# Split before fitting the tokenizer
xtr, xte, ytr, yte = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Learn vocabulary from training reviews
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(xtr)

# Convert reviews to padded sequences
xtr = pad_sequences(tokenizer.texts_to_sequences(xtr), maxlen=200)
xte = pad_sequences(tokenizer.texts_to_sequences(xte), maxlen=200)

In [ ]:
# Build model
model = Sequential([
    Input(shape=(200,)),
    Embedding(10000, 128),
    LSTM(128),
    Dense(1, activation='sigmoid')
])

# Compile and train
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    xtr, ytr,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
# Evaluate and predict
loss, acc = model.evaluate(xte, yte)
print('Test Accuracy:', round(acc * 100, 2))

i = 7
pred = model.predict(xte[i:i+1], verbose=0)[0][0]

print('Prediction:', 'Positive' if pred >= 0.5 else 'Negative')
print('Actual Label:', yte[i])